In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 02:59:21.632948: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 02:59:22.293574: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 02:59:23,295 [DEBUG] [Rain] Rain is initialized
2023-07-07 02:59:23,298 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 02:59:23,299 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 02:59:23,300 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 02:59:23,301 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 02:59:23,302 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 02:59:23,303 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


2023-07-07 02:59:23,304 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-07 02:59:23,316 [INFO] [Provisioner] provisioner is serving
2023-07-07 02:59:23,317 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 02:59:23,318 [INFO] [Coordinator] coordinator is serving
2023-07-07 02:59:23,319 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 02:59:23,323 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:59:23,324 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 02:59:23,325 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 02:59:23,326 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 02:59:23,328 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 02:59:23,329 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50152/
2023-07-07 02:59:23,331 [INFO] [Worker_50152] Worker is running on port: 50152
2023-0

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 2s 8ms/step - loss: 0.7019 - accuracy: 0.7787
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.7074 - accuracy: 0.7775
Epoch 2/5
157/157 [==============================] - 3s 10ms/step - loss: 0.7125 - accuracy: 0.7722
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.2987 - accuracy: 0.9093
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.3109 - accuracy: 0.9061
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.3034 - accuracy: 0.9092
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2315 - accuracy: 0.9293
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2407 - accuracy: 0.9299
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2373 - accuracy: 0.9295
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1923 - accura

2023-07-07 02:59:33,917 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 02:59:33,919 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 02:59:33,920 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 02:59:33,920 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 02:59:33,922 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-07 02:59:33,923 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-07 02:59:33,984 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 02:59:33,984 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from w

sending data to divider
sending data to divider
sending data to divider


2023-07-07 02:59:34,125 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-07 02:59:34,126 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker2
2023-07-07 02:59:34,127 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-07 02:59:34,149 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-07 02:59:34,151 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker3
2023-07-07 02:59:34,154 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-07 02:59:34,154 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
2023-07-07 02:59:34,158 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker1
2023-07-07 02:59:34,166 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1


...

2023-07-07 02:59:48,193 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 02:59:48,195 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider
157/157 [==============================] - 1s 6ms/step - loss: 0.0787 - accuracy: 0.9735


2023-07-07 02:59:48,210 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 02:59:48,211 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-07 02:59:48,258 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 02:59:48,266 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-07 02:59:48,271 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 02:59:48,284 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-07 02:59:48,314 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 3.
2023-07-07 02:59:48,322 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLe

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0865 - accuracy: 0.9769

Test accuracy: 97.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 02:59:48,701 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-07 02:59:48,702 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-07 02:59:48,704 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-07 02:59:48,705 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-07 02:59:48,707 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:59:48,709 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-07 02:59:48,710 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 

Epoch 1/5


Epoch 1/5
Epoch 1/5
157/157 [==============================] - 2s 6ms/step - loss: 0.1120 - accuracy: 0.9679
Epoch 2/5
157/157 [==============================] - 2s 12ms/step - loss: 0.0888 - accuracy: 0.9713
Epoch 3/5
157/157 [==============================] - 2s 12ms/step - loss: 0.0899 - accuracy: 0.9717
Epoch 4/5
157/157 [==============================] - 2s 10ms/step - loss: 0.0842 - accuracy: 0.9723
Epoch 4/5
157/157 [==============================] - 2s 10ms/step - loss: 0.0751 - accuracy: 0.9767
Epoch 4/5
157/157 [==============================] - 2s 11ms/step - loss: 0.0761 - accuracy: 0.9757
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0812 - accuracy: 0.9739
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0714 - accuracy: 0.9761
Epoch 5/5
149/157 [===========================>..] - ETA: 0s - loss: 0.0703 - accuracy: 0.9772

2023-07-07 03:00:01,169 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 03:00:01,171 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


sending data to divider
157/157 [==============================] - 1s 7ms/step - loss: 0.0707 - accuracy: 0.9775


...

DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 03:00:41,222 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


157/157 [==============================] - 1s 9ms/step - loss: 0.0618 - accuracy: 0.9806


2023-07-07 03:00:41,623 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 03:00:41,624 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 03:00:41,628 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 03:00:41,631 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1


sending data to divider


2023-07-07 03:00:41,695 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 03:00:41,733 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-07 03:00:41,736 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 03:00:41,737 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0685 - accuracy: 0.9817

Test accuracy: 98.2%
